## Reading Delta Tables

In [0]:
customers_df = spark.read.format("delta").load(
    "abfss://retailsales@harshadls01.dfs.core.windows.net/silver/customers_delta"
)

In [0]:
products_df = spark.read.format("delta").load(
    "abfss://retailsales@harshadls01.dfs.core.windows.net/silver/products_delta"
)

In [0]:
orders_df = spark.read.format("delta").load(
    "abfss://retailsales@harshadls01.dfs.core.windows.net/silver/orders_delta"
)


## Verifying Delta tables

In [0]:
display(customers_df)
display(products_df)
display(orders_df)

customer_id,customer_name,city
4,Priya,Mumbai
2,Rahul,Delhi
3,Amit,Gurgaon
1,Harsh,Noida
5,Neha,Pune


product_id,product_name,category,price
102,Mobile,Electronics,30000
101,Laptop,Electronics,50000
103,Shoes,Fashion,2000
104,Watch,Fashion,5000


order_id,customer_id,product_id,quantity
1001,1,101,1
1004,1,104,2
1002,2,102,2
1003,3,103,3
1005,5,102,1
1006,2,101,1
1007,4,104,3


## Join the Datasets

In [0]:
joined_df = (
    orders_df
    .join(customers_df, "customer_id", "inner")
    .join(products_df, "product_id", "inner")
)

display(joined_df)

product_id,customer_id,order_id,quantity,customer_name,city,product_name,category,price
101,1,1001,1,Harsh,Noida,Laptop,Electronics,50000
104,1,1004,2,Harsh,Noida,Watch,Fashion,5000
102,2,1002,2,Rahul,Delhi,Mobile,Electronics,30000
103,3,1003,3,Amit,Gurgaon,Shoes,Fashion,2000
102,5,1005,1,Neha,Pune,Mobile,Electronics,30000
101,2,1006,1,Rahul,Delhi,Laptop,Electronics,50000
104,4,1007,3,Priya,Mumbai,Watch,Fashion,5000


## Creating Revenue Column

In [0]:
from pyspark.sql.functions import col

In [0]:
sales_df = (
    joined_df
    .withColumn(
        "revenue",
        col("price") * col("quantity")
    )
)

display(sales_df)

product_id,customer_id,order_id,quantity,customer_name,city,product_name,category,price,revenue
101,1,1001,1,Harsh,Noida,Laptop,Electronics,50000,50000
104,1,1004,2,Harsh,Noida,Watch,Fashion,5000,10000
102,2,1002,2,Rahul,Delhi,Mobile,Electronics,30000,60000
103,3,1003,3,Amit,Gurgaon,Shoes,Fashion,2000,6000
102,5,1005,1,Neha,Pune,Mobile,Electronics,30000,30000
101,2,1006,1,Rahul,Delhi,Laptop,Electronics,50000,50000
104,4,1007,3,Priya,Mumbai,Watch,Fashion,5000,15000


## Gold Table 1 - Customer Revenue

In [0]:
from pyspark.sql.functions import sum

customer_revenue_df = (
    sales_df
    .groupBy(
        "customer_name",
        "city"
    )
    .agg(
        sum("revenue").alias("total_revenue")
    )
)

display(customer_revenue_df)

customer_name,city,total_revenue
Harsh,Noida,60000
Amit,Gurgaon,6000
Neha,Pune,30000
Rahul,Delhi,110000
Priya,Mumbai,15000


## Write Customer Revenue

In [0]:
customer_revenue_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://retailsales@harshadls01.dfs.core.windows.net/gold/customer_revenue"
    )

## Gold Table 2 - Category Revenue

In [0]:
category_revenue_df = (
    sales_df
    .groupBy("category")
    .agg(
        sum("revenue").alias("total_revenue")
    )
)

display(category_revenue_df)

category,total_revenue
Electronics,190000
Fashion,31000


In [0]:
category_revenue_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://retailsales@harshadls01.dfs.core.windows.net/gold/category_revenue"
    )

## Gold Table 3 - City Revenue

In [0]:
city_revenue_df = (
    sales_df
    .groupBy("city")
    .agg(
        sum("revenue").alias("total_revenue")
    )
)

display(city_revenue_df)

city,total_revenue
Delhi,110000
Pune,30000
Noida,60000
Gurgaon,6000
Mumbai,15000


In [0]:
city_revenue_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://retailsales@harshadls01.dfs.core.windows.net/gold/city_revenue"
    )

## Writing each gold dataset as a Parquet

In [0]:
customer_revenue_df.write \
    .mode("overwrite") \
    .parquet(
        "abfss://retailsales@harshadls01.dfs.core.windows.net/export/customer_revenue"
    )

In [0]:
category_revenue_df.write \
    .mode("overwrite") \
    .parquet(
        "abfss://retailsales@harshadls01.dfs.core.windows.net/export/category_revenue"
    )

In [0]:
city_revenue_df.write \
    .mode("overwrite") \
    .parquet(
        "abfss://retailsales@harshadls01.dfs.core.windows.net/export/city_revenue"
    )

In [0]:
display(customer_revenue_df)

customer_name,city,total_revenue
Harsh,Noida,60000
Amit,Gurgaon,6000
Neha,Pune,30000
Rahul,Delhi,110000
Priya,Mumbai,15000


In [0]:
export_df = spark.read.parquet(
    "abfss://retailsales@harshadls01.dfs.core.windows.net/export/customer_revenue"
)

display(export_df)
print(export_df.count())

customer_name,city,total_revenue
Harsh,Noida,60000
Amit,Gurgaon,6000
Neha,Pune,30000
Rahul,Delhi,110000
Priya,Mumbai,15000


5
